Corresponde al preprocesado de los datos obtenidos después de eliminar las filas duplicadas. Posteriormente se pasará a seleccionar un modelo adecuado para la proyección horizontal de 3 meses.

In [1]:
import pandas as pd
import matplotlib as plt
#Carguemos el csv nuevo
df = pd.read_csv("procesed_data.csv")
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12103 entries, 0 to 12102
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Unnamed: 0       12103 non-null  int64  
 1   fecha            12103 non-null  object 
 2   producto         12103 non-null  object 
 3   categoria        12103 non-null  object 
 4   cantidad         12103 non-null  int64  
 5   precio_unitario  12103 non-null  float64
 6   medida           12103 non-null  object 
 7   total            12103 non-null  float64
dtypes: float64(2), int64(2), object(4)
memory usage: 756.6+ KB
None


Hasta el momento solo habíamos comprobado que no habrían nulos y ya no tenemos filas duplicadas.
Procedemos a establecer correctamente los tipos de datos a las columnas del DF.

In [2]:
df["fecha"] = pd.to_datetime(df["fecha"])
print(df["fecha"].dtypes)

datetime64[ns]


Para analisis temporales sería útil crear columnas que contengan períodos de tiempo a partir de los datos de fecha que tenemos actualmente:

In [3]:
#Crear una columna año_mes que representa el período de tiempo de un mes
df["año_mes"]=df["fecha"].dt.to_period("M")
#Crear una columna mes que indica el número del mes con un valor entero
df["mes"] = df["fecha"].dt.month
#Creamos una columna trimestre que representa un período de mes trimestral pero con un valor entero
df["trimestre"] = df["fecha"].dt.quarter
#Finalmente creamos una columnba con el año
df["año"] = df["fecha"].dt.year
print(df.head())
print(df["año_mes"].dtypes)

   Unnamed: 0      fecha         producto  categoria  cantidad  \
0           0 2023-03-26            Pollo     Granos         5   
1           1 2023-11-06  Pasta de Tomate    Lácteos         2   
2           2 2023-07-21           Bolsas  Panadería         4   
3           3 2024-09-09            Pollo     Granos         4   
4           4 2024-10-25            Pollo     Granos         2   

   precio_unitario medida  total  año_mes  mes  trimestre   año  
0             0.95     kg   4.75  2023-03    3          1  2023  
1             1.50      u   3.00  2023-11   11          4  2023  
2             0.35      u   1.40  2023-07    7          3  2023  
3             0.95     kg   3.80  2024-09    9          3  2024  
4             0.95     kg   1.90  2024-10   10          4  2024  
period[M]


Agrupar las ventas por producto y mes, esto dado a que facilita un dataset más limpio para modelar la cantidad total ventidad por producto por mes:

In [6]:
ventas_mensuales = df.groupby(["producto","año_mes", "categoria", "medida"]).agg({"cantidad": "sum", "total":"sum"}).reset_index()

# Convertir año_mes a fecha real
ventas_mensuales["fecha"] = ventas_mensuales["año_mes"].dt.to_timestamp()
ventas_mensuales = ventas_mensuales.sort_values(by=["producto", "fecha"])

print(ventas_mensuales.head())

  producto  año_mes  categoria medida  cantidad  total      fecha
0   Bolsas  2023-03  Panadería      u       191  66.85 2023-03-01
1   Bolsas  2023-04  Panadería      u       146  51.10 2023-04-01
2   Bolsas  2023-05  Panadería      u       173  60.55 2023-05-01
3   Bolsas  2023-06  Panadería      u       172  60.20 2023-06-01
4   Bolsas  2023-07  Panadería      u       229  80.15 2023-07-01


Crear variables predictoras adicionales:

a. Variable secuencial mes_n por productos

In [ ]:
ventas_mensuales["mes_n"] = ventas_mensuales.groupby("producto").cumcount() #cuenta por grupos-
print(ventas_mensuales)hhhh               

b. Media móvil (es la media de los 3 meses anteriores):

In [8]:
ventas_mensuales["media_3m"] = (
    ventas_mensuales
    .groupby("producto")["cantidad"]
    .transform(lambda x: x.rolling(window=3, min_periods=1).mean())
)
print(ventas_mensuales)

    producto  año_mes  categoria medida  cantidad   total      fecha  mes_n  \
0     Bolsas  2023-03  Panadería      u       191   66.85 2023-03-01      0   
1     Bolsas  2023-04  Panadería      u       146   51.10 2023-04-01      1   
2     Bolsas  2023-05  Panadería      u       173   60.55 2023-05-01      2   
3     Bolsas  2023-06  Panadería      u       172   60.20 2023-06-01      3   
4     Bolsas  2023-07  Panadería      u       229   80.15 2023-07-01      4   
..       ...      ...        ...    ...       ...     ...        ...    ...   
214   Yogurt  2024-08    Bebidas      u       213  117.15 2024-08-01     17   
215   Yogurt  2024-09    Bebidas      u       198  108.90 2024-09-01     18   
216   Yogurt  2024-10    Bebidas      u       176   96.80 2024-10-01     19   
217   Yogurt  2024-11    Bebidas      u       181   99.55 2024-11-01     20   
218   Yogurt  2024-12    Bebidas      u         7    3.85 2024-12-01     21   

       media_3m  
0    191.000000  
1    168.500000

In [ ]:
Codificar variables categóricas (esto obviamente porque la mayoría de algoritmos 
necesitan valores numéricos para calcular coeficientes)

In [9]:
ventas_mensuales["producto_cod"] = ventas_mensuales["producto"].astype("category").cat.codes
ventas_mensuales["categoria_cod"] = ventas_mensuales["categoria"].astype("category").cat.codes
ventas_mensuales["medida_cod"] = ventas_mensuales["medida"].astype("category").cat.codes
print(ventas_mensuales.head())

  producto  año_mes  categoria medida  cantidad  total      fecha  mes_n  \
0   Bolsas  2023-03  Panadería      u       191  66.85 2023-03-01      0   
1   Bolsas  2023-04  Panadería      u       146  51.10 2023-04-01      1   
2   Bolsas  2023-05  Panadería      u       173  60.55 2023-05-01      2   
3   Bolsas  2023-06  Panadería      u       172  60.20 2023-06-01      3   
4   Bolsas  2023-07  Panadería      u       229  80.15 2023-07-01      4   

     media_3m  producto_cod  categoria_cod  medida_cod  
0  191.000000             0              5           1  
1  168.500000             0              5           1  
2  170.000000             0              5           1  
3  163.666667             0              5           1  
4  191.333333             0              5           1  


Guardamos

In [10]:
ventas_mensuales.to_csv("ventas_mensuales_limpio.csv", index=False)